# 1. Mechanical Refresher

* An iterator is an object that returns itself from `__iter__` and produces one next value from `__next__` until it raises `StopIteration`.

* A generator function uses `yield` to produce one value, pause with its local variables still stored, and resume from the same line the next time the caller asks for a value.

# 2. Minimal Working Example

The `for` loop calls `iter(...)` once, then repeatedly calls `__next__`.

Each call returns one value and updates `current`; when `current` passes `stop`, `StopIteration` tells the loop to stop.

In [ ]:
class CountUp:

    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self): # This object is its own iterator
        return self

    def __next__(self): # Give the next item from this iterator
        if self.current > self.stop:
            raise StopIteration # If the current number is bigger than the stop given by user, StopIteration
        value = self.current
        self.current = self.current + 1
        return value

for number in CountUp(1, 3):
    print(number)

1
2
3


# 3. Modify Drills

**Modify Drill 1.** Change the generator start and stop values, predict the produced list, then update `expected`.

In [ ]:
def count_down(start):
    while start > 0:
        yield start # Give me this value now, pause here, and continue from this exact spot when I ask for the next value
                    # Or return, but don't destroy the function's state, pause it so I can come back later
        start = start - 1

actual = []
for value in count_down(7):
    actual.append(value)
expected = [7, 6, 5, 4, 3, 2, 1]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [7, 6, 5, 4, 3, 2, 1] actual: [7, 6, 5, 4, 3, 2, 1] match: True


**Modify Drill 2.** Change the batch size and predict the yielded chunks.

In [1]:
def chunks(values, size):
    index = 0
    while index < len(values):
        yield values[index:index + size] # For start of index=0, yield values[0:1], then [2:3]...
        index = index + size

actual = []
for chunk in chunks([1, 2, 3, 4, 5], 3):
    actual.append(chunk)
expected =[[1, 2, 3], [4, 5]]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [[1, 2, 3], [4, 5]] actual: [[1, 2, 3], [4, 5]] match: True


**Modify Drill 3.** Predict when generator body prints happen relative to `next` calls.

In [2]:
def noisy_values():
    print("before first")
    yield "a"
    print("before second")
    yield "b"

stream = noisy_values()
first = next(stream) # Run/resume until you reach the next yield, then give me that value
second = next(stream)
actual = [first, second]
expected =["a", "b"]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

before first
before second
expected: ['a', 'b'] actual: ['a', 'b'] match: True


# 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.**

Break it by replacing `yield value` with `return value`. Predict why only one value can come out, then restore `yield`.

In [2]:
def first_three():
    value = 1
    while value <= 3:
        yield value # "return" would only generate one value since the function would not "yield", or "return + iterate" beyond the initial point
        value = value + 1

actual = []
for value in first_three():
    actual.append(value)
expected = [1, 2, 3]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [1, 2, 3] actual: [1, 2, 3] match: True


**Break-and-Fix Drill 2.**

Break it by deleting the state update. Predict the non-terminating loop, then restore `current = current + 1`.

In [3]:
def first_three():
    current = 1
    while current <= 3:
        yield current
        current = current + 1 # Without this state update, this function will yield 1 infinitely

actual = []
for value in first_three():
    actual.append(value)
expected = [1, 2, 3]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [1, 2, 3] actual: [1, 2, 3] match: True


# 5. Self-Verification

Expected-vs-actual prints verify finite iterator and generator outputs.

For control flow, predict which local variable changes before each `yield` and which values stay stored while the generator is paused.

# 6. Standalone Exercises

**Exercise 1.** Write a generator that yields each value doubled. Expected behavior: `[2, 4, 6]`.

In [4]:
def doubled(values):
    for value in values:
        value = value * 2
        yield value

actual = []
for value in doubled([1, 2, 3]):
    actual.append(value)
expected = [2, 4, 6]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [2, 4, 6] actual: [2, 4, 6] match: True


**Exercise 2.** Implement an iterator class for three fixed labels. Expected behavior: `['a', 'b', 'c']`.

In [7]:
class LabelIterator:

    def __init__(self):
        self.labels = ["a", "b", "c"]
        self.index = 0

    def __iter__(self): # This object is its own iterator
        return self

    def __next__(self): # Give the next item from this iterator
        if self.index >= len(self.labels):
            raise StopIteration
        label = self.labels[self.index]
        self.index = self.index + 1
        return label

actual = list(LabelIterator())
expected = ["a", "b", "c"]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: ['a', 'b', 'c'] actual: ['a', 'b', 'c'] match: True


**Exercise 3.** Use `next` manually to pull two values. Expected behavior: `[10, 20]`.

In [8]:
def values():
    yield 10
    yield 20
    yield 30

stream = values()
actual = [next(stream), next(stream)]
expected = [10, 20]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [10, 20] actual: [10, 20] match: True


**Exercise 4.** Yield rolling pairs. Expected behavior: `[(1, 2), (2, 3), (3, 4)]`.

In [9]:
def rolling_pairs(values):
    index = 0
    while index < len(values) - 1:
        yield tuple(values[index:index+2])
        index = index + 1

actual = []
for pair in rolling_pairs([1, 2, 3, 4]):
    actual.append(pair)
expected = [(1, 2), (2, 3), (3, 4)]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [(1, 2), (2, 3), (3, 4)] actual: [(1, 2), (2, 3), (3, 4)] match: True


**Exercise 5.** Stop a manual iterator correctly. Expected behavior: `[0, 1]`.

In [10]:
class TwoNumbers:

    def __init__(self):
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > 1:
            raise StopIteration
        value = self.current
        self.current = self.current + 1
        return value

actual = list(TwoNumbers())
expected = [0, 1]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [0, 1] actual: [0, 1] match: True


# 7. Applied AI/ML Drill

**ML to Python mirror:** a minibatch generator is just a lazy sequence that yields one slice of data at a time.

**Python to ML mirror:** dataset loaders and training loops rely on this same iterator/generator pattern so they can process batches without materializing every intermediate batch at once.

**Applied Drill.** Implement `minibatches` so it yields feature and target chunks lazily. Expected behavior: three batches.

In [11]:
def minibatches(features, targets, batch_size):
    index = 0
    while index < len(features):
        yield features[index:index+batch_size], targets[index:index+batch_size]
        index = index + batch_size

features = [[1], [2], [3], [4], [5]]
targets = [2, 4, 6, 8, 10]

actual = []
for batch_x, batch_y in minibatches(features, targets, 2):
    actual.append((batch_x, batch_y))
expected = [([[1], [2]], [2, 4]),
            ([[3], [4]], [6, 8]),
            ([[5]], [10])]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [([[1], [2]], [2, 4]), ([[3], [4]], [6, 8]), ([[5]], [10])] actual: [([[1], [2]], [2, 4]), ([[3], [4]], [6, 8]), ([[5]], [10])] match: True


# 8. Common Bugs

- Using `return` instead of `yield`: the function stops immediately and does not produce a stream.
- Forgetting to update loop state: the symptom is a generator or iterator that never stops.
- Reusing an exhausted iterator: the symptom is an empty second pass over the same iterator object.
- Raising `StopIteration` too early or too late: the symptom is a missing last value or an extra bad value.

# 9. Compounding Drill

Combine generators with Chapter 1 function objects: lazily transform values with a function passed into the generator.

In [12]:
def transform_stream(values, transform_fn):

    for value in values:
        yield transform_fn(value)

def square(x):
    return x * x

actual = []
for value in transform_stream([2, 3, 4], square):
    actual.append(value)
expected = [4, 9, 16]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [4, 9, 16] actual: [4, 9, 16] match: True


# 10. Chapter Project

No chapter project in this chapter. Projects appear every 2-3 chapters so the longer drills can compound several mechanics at once.